In [1]:
!pip uninstall -y transformers
!pip install --upgrade --no-cache-dir transformers

# 🧪 Step 2: Verify import and version
try:
    from transformers import AutoTokenizer
    import transformers
    print(f" AutoTokenizer imported successfully from transformers v{transformers.__version__}")
except ImportError as e:
    print(f" ImportError: {e}")



Found existing installation: transformers 4.55.2
Uninstalling transformers-4.55.2:
  Successfully uninstalled transformers-4.55.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 133.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 293.4 MB/s eta 0:00:00
 AutoTokenizer imported successfully from transformers v4.55.4


In [2]:
%pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 73.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 45.1 MB/s eta 0:00:00


In [3]:
# ⬇️ Install required packages

!pip install bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 34.5 MB/s eta 0:00:00


In [4]:
!pip install -q streamlit pyngrok
!pip install bitsandbytes
!pip install --upgrade transformers


# 📝 Save your Streamlit app
with open("app.py", "w") as f:
    f.write("""

import streamlit as st
import os
import json
from peft import PeftModel
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
from google.colab import drive
import re
import gc

torch.cuda.empty_cache()
gc.collect()
if 'base_model' in locals() or 'base_model' in globals(): del base_model
if 'ft_model' in locals() or 'ft_model' in globals(): del ft_model
torch.cuda.empty_cache()
gc.collect()

BASE_MODEL_SAVE_DIR = "/content/drive/MyDrive/capstone/mistral_bpr_model"
FT_MODEL_SAVE_DIR = "/content/drive/MyDrive/capstone/mistral_dpo_lora"

USE_GPU = torch.cuda.is_available()
tokenizer_base = AutoTokenizer.from_pretrained(BASE_MODEL_SAVE_DIR)
tokenizer_ft = AutoTokenizer.from_pretrained(FT_MODEL_SAVE_DIR)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)


base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_SAVE_DIR,
    torch_dtype=torch.float16 if USE_GPU else torch.float32,
    device_map="auto",
    quantization_config=bnb_config,
    low_cpu_mem_usage=True
)

#base_model = base_model.to_empty(device=torch.device("cuda"))


#ft_model = AutoModelForCausalLM.from_pretrained(
#    FT_MODEL_SAVE_DIR,
#    torch_dtype=torch.float16 if USE_GPU else torch.float32,
#    device_map="auto",
#    quantization_config=bnb_config,
#    low_cpu_mem_usage=True
#)

#ft_model = ft_model.to_empty(device=torch.device("cuda"))

ft_model = PeftModel.from_pretrained(base_model, FT_MODEL_SAVE_DIR)

text_generator_base = pipeline(
    "text-generation",
    model=base_model,
    tokenizer=tokenizer_base,
    device_map="auto",
    max_new_tokens=160,
    do_sample=True,
    temperature=0.7,
    top_p=0.92,
    repetition_penalty=1.05,
    no_repeat_ngram_size=3,
)

text_generator_ft = pipeline(
    "text-generation",
    model=ft_model,
    tokenizer=tokenizer_ft,
    device_map="auto",
    max_new_tokens=160,
    do_sample=True,
    temperature=0.7,
    top_p=0.92,
    repetition_penalty=1.05,
    no_repeat_ngram_size=3,
)

def strip_prompt_from_response(prompt, response):

    prompt_clean = prompt.strip()
    response_clean = response.strip()
    if response_clean.startswith(prompt_clean):
        return response_clean[len(prompt_clean):].strip()
    return response_clean

def get_base_model_response(prompt):
    response = text_generator_base(prompt)[0]["generated_text"].strip()
    response_pure = strip_prompt_from_response(prompt, response)
    return "Base model response: " + response_pure

def get_finetuned_model_response(prompt):
    response = text_generator_ft(prompt)[0]["generated_text"].strip()
    response_pure = strip_prompt_from_response(prompt, response)
    return "Fine-tuned model response: " + response_pure

def calculate_bpr_and_toxicity(response):
    return {
        "BPR": round(len(response) % 10 / 10, 2),
        "Toxicity": round((len(response) % 5) / 5, 2)
    }

st.set_page_config(page_title="ReligionCARE Bias Dashboard", layout="wide")
st.title("ReligionCARE Comparison Dashboard")

prompt = st.text_input("Enter a prompt to evaluate:")

if prompt:
    col1, col2 = st.columns(2)

    with col1:
        st.subheader("Base Model Response")
        base_response = get_base_model_response(prompt)
        st.write(base_response)
        #base_scores = calculate_bpr_and_toxicity(base_response)
        #st.metric("Bias Pair Ratio (BPR)", base_scores["BPR"])
        #st.metric("Toxicity", base_scores["Toxicity"])

    with col2:
        st.subheader("Fine-Tuned Model Response")
        finetuned_response = get_finetuned_model_response(prompt)
        st.write(finetuned_response)
        #finetuned_scores = calculate_bpr_and_toxicity(finetuned_response)
        #st.metric("Bias Pair Ratio (BPR)", finetuned_scores["BPR"])
        #st.metric("Toxicity", finetuned_scores["Toxicity"])

""")

#  Load ngrok token from Colab Secrets
import os
from pyngrok import conf, ngrok
NGROK_AUTH_TOKEN = '315SD04xZJeQLH78B0FKKSOnl2c_48g88mxtvREdBTpQi1sYV'  # Colab Secrets
conf.get_default().auth_token = NGROK_AUTH_TOKEN

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

#  Launch Streamlit in a background thread
import threading, time

def run_streamlit():
    !streamlit run app.py --server.port 8501 --server.enableCORS false --server.enableXsrfProtection false

threading.Thread(target=run_streamlit).start()
time.sleep(5)  # Wait for Streamlit to boot

#  Create public tunnel
#public_url = ngrok.connect(addr=8501)
public_url = ngrok.connect(addr=8501, proto="http", hostname="javelin-relieved-personally.ngrok-free.app")

print(f"🔗 Your Streamlit dashboard is live at: {public_url}")

Mounted at /content/drive



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.124.186.12:8501

🔗 Your Streamlit dashboard is live at: NgrokTunnel: "https://javelin-relieved-personally.ngrok-free.app" -> "http://localhost:8501"
